<a href="https://colab.research.google.com/github/justorfc/Estadistica_Aplicada_con_Python_y_R/blob/main/8_Semana_8_Comparaci%C3%B3n_M%C3%BAltiple_de_Distribuciones_y_Proyecto_Parcial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Esta es la propuesta estructurada para la **Semana 8**, que concluye el Eje III. En esta semana se eleva el nivel analítico: ya no evaluamos un solo modelo, sino que competimos múltiples distribuciones simultáneamente y utilizamos criterios de selección de modelos (como el Criterio de Información de Akaike - AIC) para tomar una decisión fundamentada, culminando en el **Proyecto Parcial de Distribuciones**.

### Semana 8: Comparación Múltiple de Distribuciones y Proyecto Parcial

**Resultado de aprendizaje:** Ajusta y compara simultáneamente distribuciones teóricas (Lognormal, Gamma, Pearson) evaluando métricas de bondad de ajuste y criterios de información (AIC/BIC) para sustentar el Proyecto Parcial.

---

#### Sesión 1: Comparación Avanzada y Selección de Modelos (80 - 90 minutos)

**Objetivo:** Comprender que varios modelos pueden parecer "buenos" visualmente, y utilizar la verosimilitud y el criterio AIC en Python para desempatar objetivamente al analizar datos de caudales.

* **20 min - Diálogo socrático y conceptualización (Lápiz y papel):**
* *Situación:* El río principal del distrito de riego se desborda en invierno. Tenemos registros de caudales máximos anuales. ¿Qué pasa si tanto la distribución Gamma como la Lognormal pasan la prueba de Kolmogorov-Smirnov? ¿Cómo elegimos la mejor para diseñar el muro de contención?
* *Actividad:* Explicación intuitiva de la "Verosimilitud" (Likelihood) y el principio de parsimonia penalizado por el Criterio de Información de Akaike (AIC).


* **45 min - Exploración en Google Colab (Python):**
* Carga del cuaderno de la semana 8.
* Ajuste simultáneo de distribuciones Normal, Lognormal y Gamma a una serie de caudales.
* Cálculo de la Función de Log-Verosimilitud (`logpdf`) y el AIC para seleccionar el modelo ganador.


* **15 min - Reflexión manuscrita:**
* Cierre del análisis: Justificación de por qué en hidrología preferimos distribuciones con sesgo positivo (colas pesadas a la derecha) para caudales.



---

#### Sesión 2: Desarrollo del Proyecto Parcial en R (80 - 90 minutos)

**Objetivo:** Consolidar el Eje III utilizando las potentes herramientas de comparación del paquete `fitdistrplus` en R, generando el informe reproducible final (Proyecto Parcial).

* **20 min - El poder de `fitdistrplus` para comparación múltiple:**
* Explicación de cómo las funciones `denscomp()`, `qqcomp()`, `cdfcomp()` y `ppcomp()` permiten graficar múltiples distribuciones en un solo lienzo en R.


* **25 min - Prompts para selección de modelos en R:**
* Demostración de cómo pedir al asistente de IA que use la función `gofstat()` de `fitdistrplus` para extraer de inmediato las pruebas KS, Anderson-Darling, AIC y BIC en una tabla resumen elegante.


* **40 min - Reto en Posit Cloud (Proyecto Parcial):**
* Los estudiantes ejecutan el flujo completo con sus propios datos o los simulados, exportan el informe en RMarkdown/Quarto, y presentan una defensa breve (3-5 minutos) de su selección ante el docente, evaluando sus bitácoras de IA.



---

A continuación, el contenido para que lo integres en las celdas de tu cuaderno de Google Colab.

---

### Celda de Texto 1

```markdown
# Semana 8: Comparación Múltiple de Distribuciones (AIC/BIC)
**Asignatura:** Estadística Aplicada con Python y R  
**Programa:** Ingeniería Agrícola - Universidad de Sucre  
**Profesor:** Justo Rafael Fuentes Cuello  

---

### Situación de Interés: El Desempate Hidrológico
Cuando diseñamos obras de protección contra inundaciones, analizamos los **Caudales Máximos Anuales**. Es común que, al ajustar modelos, tanto la distribución **Gamma** como la **Lognormal** se vean bien en el gráfico y ambas pasen la prueba de Kolmogorov-Smirnov.

¿Cómo desempatamos? Hoy aprenderemos a calcular el **Criterio de Información de Akaike (AIC)**. Esta métrica premia al modelo que mejor se ajusta a los datos (Máxima Verosimilitud), pero lo penaliza si es demasiado complejo matemáticamente. **La regla de oro: El modelo con el AIC más bajo gana.**

```

### Celda de Código 1

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

sns.set_theme(style="ticks")
np.random.seed(101)

# Simulamos 60 años de Caudales Máximos Anuales (m3/s)
# Usaremos una distribución Lognormal de fondo para la simulación
caudales_maximos = stats.lognorm.rvs(s=0.5, scale=120, size=60)

print(f"Registro hidrológico: 60 años generados.")
print(f"Caudal promedio histórico: {caudales_maximos.mean():.1f} m³/s")
print(f"Caudal máximo extremo registrado: {caudales_maximos.max():.1f} m³/s")

### 1. Ajuste Simultáneo de Tres Modelos

Vamos a competir tres "plantillas" matemáticas:
1. **Normal:** (Sabemos que fallará, pero servirá como línea base).
2. **Gamma:** Muy usada en hidrología.
3. **Lognormal:** Especial para variables estrictamente positivas con varianza alta.

```

### Celda de Código 2

In [ ]:
# 1. Ajuste Normal
mu_norm, std_norm = stats.norm.fit(caudales_maximos)

# 2. Ajuste Gamma
a_gamma, loc_gamma, scale_gamma = stats.gamma.fit(caudales_maximos)

# 3. Ajuste Lognormal (s=shape, loc, scale)
s_log, loc_log, scale_log = stats.lognorm.fit(caudales_maximos)

# Graficamos la competencia visual
plt.figure(figsize=(9, 5))
sns.histplot(caudales_maximos, stat='density', color='lightblue', label='Caudales Empíricos')

x = np.linspace(0, 400, 200)
plt.plot(x, stats.norm.pdf(x, mu_norm, std_norm), 'r:', lw=2, label='Normal (Base)')
plt.plot(x, stats.gamma.pdf(x, a_gamma, loc_gamma, scale_gamma), 'g--', lw=2.5, label='Gamma')
plt.plot(x, stats.lognorm.pdf(x, s_log, loc_log, scale_log), 'k-', lw=2, label='Lognormal')

plt.title('Competencia de Densidades: Normal vs Gamma vs Lognormal')
plt.xlabel('Caudal Máximo (m³/s)')
plt.ylabel('Densidad')
plt.legend()
plt.show()

### 2. Cuantificación del Ajuste: Log-Verosimilitud y AIC

Visualmente, la Gamma y la Lognormal están empatadas.
Vamos a calcular el AIC usando la fórmula:
**$AIC = 2k - 2 \cdot \ln(L)$**
Donde $k$ es el número de parámetros de la distribución y $\ln(L)$ es el logaritmo de la verosimilitud (Log-Likelihood).

```

### Celda de Código 3

In [ ]:
def calcular_aic(dist, datos, parametros):
    """Función para calcular la Log-Verosimilitud y el AIC de un ajuste"""
    k = len(parametros) # Número de parámetros
    log_verosimilitud = np.sum(dist.logpdf(datos, *parametros))
    aic = 2 * k - 2 * log_verosimilitud
    return aic

# Calculamos AIC para los 3 modelos
aic_norm = calcular_aic(stats.norm, caudales_maximos, (mu_norm, std_norm))
aic_gamma = calcular_aic(stats.gamma, caudales_maximos, (a_gamma, loc_gamma, scale_gamma))
aic_lognorm = calcular_aic(stats.lognorm, caudales_maximos, (s_log, loc_log, scale_log))

# Creamos una tabla comparativa
tabla_resultados = pd.DataFrame({
    'Distribución': ['Normal', 'Gamma', 'Lognormal'],
    'AIC': [aic_norm, aic_gamma, aic_lognorm]
})

# Ordenamos de menor a mayor (el más bajo es el mejor)
tabla_resultados = tabla_resultados.sort_values(by='AIC').reset_index(drop=True)
print("--- Tabla de Criterio de Información de Akaike (AIC) ---")
display(tabla_resultados)

ganador = tabla_resultados.loc[0, 'Distribución']
print(f"\n🏆 Según el AIC, la mejor distribución matemática para estos caudales es la: {ganador}")

### 3. Predicción Ingenieril (Periodo de Retorno)

Una vez elegido el modelo ganador (Lognormal en este caso), podemos calcular el caudal de diseño.

Si queremos construir un puente que no se caiga en un evento que ocurre, en promedio, una vez cada 100 años (Periodo de Retorno $T = 100$ años), necesitamos calcular el cuantil correspondiente al $99\%$ de probabilidad acumulada ($1 - 1/T = 0.99$).

```

### Celda de Código 4

In [ ]:
# Usamos el método ppf (cuantil) del modelo ganador (Lognormal)
probabilidad_acumulada_100_anios = 1 - (1/100) # 0.99
caudal_diseno_100a = stats.lognorm.ppf(probabilidad_acumulada_100_anios, s_log, loc_log, scale_log)

print(f"Caudal de Diseño para Tr = 100 años: {caudal_diseno_100a:.2f} m³/s")

# Dibujamos esto en la curva
plt.figure(figsize=(7, 4))
plt.plot(x, stats.lognorm.pdf(x, s_log, loc_log, scale_log), 'k-', lw=2)
plt.fill_between(x, stats.lognorm.pdf(x, s_log, loc_log, scale_log), where=(x >= caudal_diseno_100a), color='red', alpha=0.5)
plt.axvline(caudal_diseno_100a, color='red', linestyle='--')
plt.title(f'Área de Excedencia (1%) - Caudal: {caudal_diseno_100a:.1f} m³/s')
plt.xlabel('Caudal Máximo (m³/s)')
plt.show()

### 🛑 Reflexión y Reserva Cognitiva (Síntesis manuscrita)
Con lápiz y papel, consolida tus apuntes del Eje III:
1. Explica con tus palabras por qué usamos el AIC para desempatar modelos estadísticos en lugar de confiar únicamente en nuestra percepción del histograma.
2. Si un ingeniero rival diseñara el mismo puente utilizando la distribución Normal basándose en los parámetros de la primera celda, ¿el puente correría riesgo de caerse? (Piensa en cómo la cola derecha de la Normal subestima los valores extremos respecto a la Lognormal).
3. ¿Qué significa hidrológicamente el concepto de "Periodo de Retorno de 100 años"? (Ojo: ¡no significa que ocurrirá exactamente dentro de 100 años!).

---

### Instrucciones para el reto en R (PROYECTO PARCIAL DE DISTRIBUCIONES)

**Misión Final del Eje III:** Has realizado la comparación matemáticamente desde cero en Python. En **R**, la librería `fitdistrplus` está diseñada exactamente para automatizar este flujo hidrológico.

**Pasos a seguir:**
1. Abre un nuevo proyecto de RMarkdown en Posit Cloud. Este será tu informe entregable (Proyecto Parcial).
2. Utiliza este *prompt* maestro con tu agente de IA:
   > *"Actúa como un ingeniero hidrólogo programando en R. En Python ajusté caudales máximos a las distribuciones Normal, Gamma y Lognormal, calculé el AIC manualmente y calculé el cuantil del 99% para un periodo de retorno de 100 años. Necesito presentar mi Proyecto Parcial en RMarkdown usando la librería `fitdistrplus`. Genera un script que: 1) Simule los datos. 2) Ajuste simultáneamente las 3 distribuciones usando `fitdist()`. 3) Use `denscomp()`, `qqcomp()` y `cdfcomp()` para compararlas gráficamente. 4) Use `gofstat()` para mostrar la tabla con los estadísticos de bondad de ajuste (incluyendo AIC). 5) Calcule el cuantil 99% con la función de la distribución ganadora (ej. `qlnorm`). Documenta el código paso a paso."*
3. Observa cómo `gofstat(list(fit_norm, fit_gamma, fit_lognorm))` en R te devuelve instantáneamente las pruebas de Kolmogorov-Smirnov, Anderson-Darling, Cramer-von Mises, AIC y BIC en una sola salida de texto estructurada.
4. **Entrega y Sustentación:** Renderiza tu documento. Asegúrate de incluir tus conclusiones técnicas y tu "Bitácora de IA". Estarás listo para presentar y defender verbalmente por qué elegiste tu modelo ganador durante la clase.